# Part C — Understanding Tools: Contracts, Validation and Safe Execution

**Prerequisites:** Part A completed (the database and `books_api.py` exist). Part B's Qwen model **is not needed yet**; we'll use it in Part D. Continue here after Part B in your running Colab notebook.

**Learning outcomes**
- Explain how an ordinary Python function differs from a registered LLM tool.
- Inspect **name**, **description**, and **JSON input schema** presented to a model.
- Invoke a tool manually without an LLM and inspect its output.
- Explain schema-based validation, Boolean conversion, and why purchasing is not enabled yet.

**Important:** The Python script files are created with `%%writefile`. Scripts that only read the database run in subprocesses using `!python` to keep their output easy to inspect. No model is used in this part.

## C1 — Install LangChain's tool primitives

We install only `langchain-core` and Pydantic here. A **tool definition** is not an agent or LLM by itself. We'll explore the agent loop later in Part E.

**Before running:** Ensure `books_api.py` and `bookstore.db` from Part A are in the current working directory.

In [ ]:
%%writefile c_install.py
"""Part C1: install only the tool/schema libraries used by these sections."""
import subprocess
import sys

packages = ["langchain-core>=1.0,<2", "pydantic>=2,<3"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
print("Installed:", ", ".join(packages))
print("If langchain_core was imported before an upgrade, restart the runtime.")


In [ ]:
!python c_install.py

**Questions C1**

1. **What** is LangChain's role in this section? Are we using a model yet?
2. **Why** is a formal input schema useful compared with asking the model to guess function arguments?
3. **How** is package installation (`!python`) different from executing a script with persistent model objects (`%run -i`)?

## C2 — Register two read-only tools

The API functions `books_api.search_books` and `books_api.get_book_rating` already work. We wrap them with `@tool`, attach Pydantic argument schemas, and expose only a small allowlist.

- `search_books` can filter the catalog. **It does not provide ratings.**
- `get_book_rating` can retrieve the rating for a particular book ID.
- We deliberately **do not expose** `order_book` / `buy_book`: it modifies the database and should require an approval mechanism before it can be used by an agent.

A model receives **descriptions and argument schemas**, not the body of our SQL query. The tool executor is responsible for running the Python function and returning a result.

In [ ]:
%%writefile bookstore_tools.py
"""Part C2 — Expose Part A's read-only Python functions as LangChain tools.

The tool names and schemas are visible to the LLM, not Python's SQL code.
Deliberately excludes order_book: actions that change the DB are not offered
until we add a human-approval workflow in a later part.
"""
import json
from typing import Optional

from langchain_core.tools import tool
from pydantic import BaseModel, ConfigDict, Field

import books_api


class SearchBooksArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    query: Optional[str] = Field(None, description="Text contained in the book title or author name.")
    genre: Optional[str] = Field(None, description="Genre such as sci-fi, fantasy or mystery.")
    max_price: Optional[float] = Field(None, ge=0, description="Maximum price in USD, inclusive.")
    is_bestseller: Optional[bool] = Field(None, description="True for bestsellers, False for non-bestsellers. Omit for both.")
    limit: int = Field(10, ge=1, le=100, description="Maximum number of books to return.")


class BookRatingArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    book_id: int = Field(..., gt=0, strict=True, description="Positive integer catalog ID, not a title.")


@tool("search_books", args_schema=SearchBooksArgs)
def search_books_tool(
    query: Optional[str] = None,
    genre: Optional[str] = None,
    max_price: Optional[float] = None,
    is_bestseller: Optional[bool] = None,
    limit: int = 10,
) -> str:
    """Search books using optional filters; returns JSON rows with book IDs and prices. Does not return review ratings."""
    matches = books_api.search_books(
        query=query,
        genre=genre,
        max_price=max_price,
        is_bestseller=is_bestseller,
        limit=limit,
    )
    return json.dumps(matches, ensure_ascii=False)


@tool("get_book_rating", args_schema=BookRatingArgs)
def get_book_rating_tool(book_id: int) -> str:
    """Look up average review rating and review count for one book ID. Reports missing and unrated books distinctly."""
    return json.dumps(books_api.get_book_rating(book_id), ensure_ascii=False)


# The registry is an explicit read-only allowlist, not arbitrary Python execution.
READ_ONLY_TOOLS = {t.name: t for t in (search_books_tool, get_book_rating_tool)}


def make_tool_schemas():
    """Tool contracts in Hugging Face's standard JSON schema format."""
    return [{
        "type": "function",
        "function": {
            "name": item.name,
            "description": item.description,
            "parameters": item.args_schema.model_json_schema(),
        },
    } for item in READ_ONLY_TOOLS.values()]


TOOL_SCHEMAS = make_tool_schemas()


**Questions C2 — predict before executing anything**

1. **What** does `@tool` add to a Python function?
2. **Why** is the tool description as important as its name?
3. **How** does `Optional[bool]` differ from `str` for `is_bestseller`?
4. **What** should happen if the model provides `is_bestseller="False"`? What should happen if it provides `"maybe"`?
5. **Why** does the tool return JSON-encoded data rather than an arbitrary human-style paragraph?

## C3 — Inspect schema and invoke tools manually

Next, we ask Python—not Qwen—to print the tool contracts and call the functions. Compare the plain API in Part A with `.invoke(...)` here.

**Predict:** Which fields are optional? Which tool uses `book_id`? What happens with a nonexistent book?

In [ ]:
%%writefile c_inspect_tools.py
"""Part C3 — Inspect tool contracts and invoke them directly (NO MODEL)."""
import json
from bookstore_tools import READ_ONLY_TOOLS, TOOL_SCHEMAS, SearchBooksArgs

print("Registered read-only tool names:", list(READ_ONLY_TOOLS))
print("\nTool schemas the LLM is allowed to see:")
print(json.dumps(TOOL_SCHEMAS, indent=2, ensure_ascii=False))

search_tool = READ_ONLY_TOOLS["search_books"]
rating_tool = READ_ONLY_TOOLS["get_book_rating"]
print("\nDirect .invoke(...) — not a model-generated tool call:")
result = search_tool.invoke({"genre": "sci-fi", "max_price": 16, "is_bestseller": True})
print("Search result:", json.loads(result))
print("Dune rating:", json.loads(rating_tool.invoke({"book_id": 1})))

# Pydantic turns a common model error into the *Boolean* False.
normalized = SearchBooksArgs.model_validate({"genre": "sci-fi", "is_bestseller": "False"})
print("\nInput text 'False' normalized into:", repr(normalized.is_bestseller),
      "type:", type(normalized.is_bestseller).__name__)
print("Non-bestsellers:", json.loads(search_tool.invoke({"genre": "sci-fi", "is_bestseller": "False"})))
print("\nIMPORTANT: no language model has been used in Part C.")


In [ ]:
!python c_inspect_tools.py

**Questions C3**

1. **What** part of the printed output is the schema? Which part is actual database content?
2. **Who** chose the tool in this example: the LLM or the programmer?
3. **How** do you tell the difference between `True`, `False`, and a missing bestseller filter?
4. **Why** does `search_books` return multiple books but `get_book_rating` return one record?

## C4 — Run reproducible tool checks

These checks confirm that valid tool inputs work, invalid arguments are rejected, the Boolean-string case is corrected, and **no new order** is inserted in SQLite.

In [ ]:
%%writefile c_tool_checks.py
"""Part C4 — Offline assertions for tools, validation and database safety."""
import json
import sqlite3
from contextlib import closing
from bookstore_tools import READ_ONLY_TOOLS, SearchBooksArgs
import books_api

def order_count():
    with closing(sqlite3.connect(books_api.DB_PATH)) as db:
        return db.execute("SELECT COUNT(*) FROM orders").fetchone()[0]

initial = order_count()
catalog = json.loads(READ_ONLY_TOOLS["search_books"].invoke(
    {"genre": "sci-fi", "max_price": 16, "is_bestseller": True}))
assert any(b["title"] == "Dune" for b in catalog)
assert all(b["price"] <= 16 and b["is_bestseller"] for b in catalog)

nonbest = json.loads(READ_ONLY_TOOLS["search_books"].invoke(
    {"genre": "sci-fi", "is_bestseller": "False"}))
assert all(not b["is_bestseller"] for b in nonbest)
assert SearchBooksArgs.model_validate({"is_bestseller": "False"}).is_bestseller is False

rating = json.loads(READ_ONLY_TOOLS["get_book_rating"].invoke({"book_id": 1}))
assert rating["found"] and rating["review_count"] > 0

for bad in ({"max_price": -1}, {"is_bestseller": "maybe"}, {"surprise": 42}):
    try:
        READ_ONLY_TOOLS["search_books"].invoke(bad)
    except Exception as exc:
        print("Invalid input rejected:", bad, "->", type(exc).__name__)
    else:
        raise AssertionError(f"Tool accepted invalid arguments: {bad}")
assert set(READ_ONLY_TOOLS) == {"search_books", "get_book_rating"}
assert order_count() == initial, "A read-only tool should not change orders!"
print("PASS: real SQLite tool calls, correct Boolean, validation, and no new orders.")


In [ ]:
!python c_tool_checks.py

**Part C practical exercises**

1. **Inspect:** Change the `max_price` argument in `c_inspect_tools.py` from `16` to `10`. Rerun the writefile and execution cells. Explain the result.
2. **Validate:** In `c_tool_checks.py`, add a negative test for `book_id=0`. Why does the schema reject it?
3. **Design:** Draft a JSON schema (on paper or in Markdown) for a **new read-only** `get_book_details(book_id)` function. What fields would it return? *Do not implement it yet; Part F is for extending the agent.*
4. **Discuss:** Should an LLM be allowed to place an order whenever a tool call is syntactically valid? Why not?

**Exit ticket:** A tool definition tells the model **what can be requested**; a Python executor determines **what actually happens**.